In [1]:
!git clone https://github.com/cszn/DPIR.git
%cd DPIR
# !pip install -r requirements.txt -q
print("DPIR 설치 완료")

Cloning into 'DPIR'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 239 (delta 48), reused 35 (delta 35), pack-reused 179 (from 1)
Receiving objects: 100% (239/239), 6.85 MiB | 15.31 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/DPIR
DPIR 설치 완료


In [2]:
import os
os.makedirs("model_zoo", exist_ok=True)

# DPIR deblur 모델 (grayscale)
!wget -q "https://github.com/cszn/KAIR/releases/download/v1.0/drunet_gray.pth" \
     -O "model_zoo/drunet_gray.pth"

size = os.path.getsize("model_zoo/drunet_gray.pth")
print(f"모델 다운로드 완료 ({size/1024/1024:.1f} MB)")

모델 다운로드 완료 (124.5 MB)


In [3]:
from google.colab import files
import os

os.makedirs("inputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("original.png, deg.png 업로드하세요 (Gaussian 블러 버전)")
uploaded = files.upload()
for fname in uploaded:
    with open(f"inputs/{fname}", "wb") as f:
        f.write(uploaded[fname])
    print(f"{fname} 업로드 완료")

# 확인
import cv2
deg = cv2.imread("inputs/deg.png", 0)
original = cv2.imread("inputs/original.png", 0)
psnr = cv2.PSNR(original, deg)
print(f"현재 deg.png PSNR: {psnr:.2f} dB")

original.png, deg.png 업로드하세요 (Gaussian 블러 버전)


Saving deg.png to deg.png
Saving original.png to original.png
deg.png 업로드 완료
original.png 업로드 완료
현재 deg.png PSNR: 24.38 dB


In [4]:
import torch
import numpy as np
import cv2
import sys
sys.path.insert(0, '/content/DPIR')

from models.network_unet import UNetRes as DRUNet
from utils import utils_model, utils_pnp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 로드
model = DRUNet(in_nc=2, out_nc=1, nc=[64,128,256,512], nb=4, act_mode='R',
               downsample_mode='strideconv', upsample_mode='convtranspose')
model.load_state_dict(torch.load('model_zoo/drunet_gray.pth', map_location=device), strict=True)
model = model.to(device).eval()
print("DRUNet 모델 로드 완료")

# Gaussian PSF 생성 (phase1.py와 동일)
ksize, sigma_psf = 21, 4.0
k1d = cv2.getGaussianKernel(ksize, sigma_psf)
psf = (k1d @ k1d.T).astype(np.float64)
psf /= psf.sum()

# 입력 이미지
deg = cv2.imread("inputs/deg.png", 0).astype(np.float32) / 255.0

# DPIR 파라미터
noise_level = 3.0 / 255.0  # phase1.py와 동일
# iter_num과 rho만 조정
iter_num = 24   # 8 -> 24 -> 40 -> 24
rho = 0.05      # 0.1 -> 0.05 -> 0.03 -> 0.05

# PSF FFT
def psf_to_fft(psf, size):
    pad = np.zeros((size, size), dtype=np.float64)
    h, w = psf.shape
    cy, cx = size//2, size//2
    pad[cy-h//2:cy+h//2+1, cx-w//2:cx+w//2+1] = psf
    pad = np.fft.ifftshift(pad)
    return np.fft.fft2(pad)

H = psf_to_fft(psf, deg.shape[0])
HT = np.conj(H)
HTH = np.abs(H)**2

# DPIR (HQS 반복)
x = deg.copy()
for i in range(iter_num):
    # Data fidelity step (Wiener-like)
    sigma_d = noise_level / np.sqrt(rho * (i+1))
    G = np.fft.fft2(x)
    F = (HT * np.fft.fft2(deg) + rho * G) / (HTH + rho)
    x = np.real(np.fft.ifft2(F))
    x = np.clip(x, 0, 1)

    # Denoiser step (DRUNet)
    sigma_n = noise_level / np.sqrt(rho * (i+1))
    inp = torch.from_numpy(x).float().unsqueeze(0).unsqueeze(0)
    sigma_map = torch.ones_like(inp) * sigma_n
    inp_cat = torch.cat([inp, sigma_map], dim=1).to(device)
    with torch.no_grad():
        x = model(inp_cat).squeeze().cpu().numpy()
    x = np.clip(x, 0, 1)

out = (x * 255).astype(np.uint8)
cv2.imwrite("outputs/dpir_restored.png", out)
print("DPIR 복원 완료")

# 비교표
from skimage.metrics import structural_similarity as ssim
original = cv2.imread("inputs/original.png", 0)
degraded = cv2.imread("inputs/deg.png", 0)

psnr_deg   = cv2.PSNR(original, degraded)
psnr_dpir  = cv2.PSNR(original, out)
ssim_deg   = ssim(original, degraded)
ssim_dpir  = ssim(original, out)
psnr_wiener, ssim_wiener = 26.95, 0.7225  # Phase 1 결과

print("\n══ 최종 비교표 ════════════════════════════")
print(f"{'방법':<16} {'PSNR (dB)':>10} {'SSIM':>8}")
print("─" * 38)
print(f"{'블러 (기준선)':<16} {psnr_deg:>10.2f} {ssim_deg:>8.4f}")
print(f"{'Wiener Filter':<16} {psnr_wiener:>10.2f} {ssim_wiener:>8.4f}")
print(f"{'DPIR (DL)':<16} {psnr_dpir:>10.2f} {ssim_dpir:>8.4f}")
print("─" * 38)
print(f"Wiener 대비 DPIR: {psnr_dpir - psnr_wiener:+.2f} dB")

DRUNet 모델 로드 완료
DPIR 복원 완료

══ 최종 비교표 ════════════════════════════
방법                PSNR (dB)     SSIM
──────────────────────────────────────
블러 (기준선)              24.38   0.6380
Wiener Filter         26.95   0.7225
DPIR (DL)             27.52   0.7652
──────────────────────────────────────
Wiener 대비 DPIR: +0.57 dB


In [6]:
# Wiener 이미지 업로드 후 4장 비교 이미지 생성
from google.colab import files
import cv2
import numpy as np
import os

os.makedirs("outputs", exist_ok=True)

print("wiener_restored.png 업로드")
uploaded = files.upload()
for fname in uploaded:
    with open(f"outputs/{fname}", "wb") as f:
        f.write(uploaded[fname])
    print(f"{fname} 업로드 완료")

# 4장 로드
original = cv2.imread("inputs/original.png", 0)
degraded = cv2.imread("inputs/deg.png", 0)
wiener   = cv2.imread("outputs/wiener_restored.png", 0)
dpir     = cv2.imread("outputs/dpir_restored.png", 0)

# 라벨 추가 함수
def add_label(img, text, psnr=None, ssim=None):
    out = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    label = text
    if psnr: label += f"  PSNR: {psnr:.2f}dB"
    if ssim: label += f"  SSIM: {ssim:.4f}"
    cv2.putText(out, label, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return out

orig_l   = add_label(original, "Original")
deg_l    = add_label(degraded, "Blurred",       psnr=24.38, ssim=0.6380)
wiener_l = add_label(wiener,   "Wiener Filter", psnr=26.95, ssim=0.7225)
dpir_l   = add_label(dpir,     "DPIR (DL)",     psnr=27.52, ssim=0.7652)

# 4장 가로 합치기
gap = np.ones((original.shape[0], 8, 3), dtype=np.uint8) * 180
comparison = np.hstack([orig_l, gap, deg_l, gap, wiener_l, gap, dpir_l])

cv2.imwrite("outputs/comparison_4panel.png", comparison)
print("4장 비교 이미지 생성 완료")

# 다운로드
files.download("outputs/comparison_4panel.png")
files.download("outputs/dpir_restored.png")
print("다운로드 완료")

wiener_restored.png 업로드


Saving wiener_restored.png to wiener_restored.png
wiener_restored.png 업로드 완료
4장 비교 이미지 생성 완료


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드 완료
